# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

In [9]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv")

In [10]:
def clean_data(df):
    
    # Rename columns
    df = df.rename(columns={
        'ST': 'state',
        'Customer Lifetime Value': 'customer_lifetime_value',
        'Monthly Premium Auto': 'monthly_premium_auto',
        'Number of Open Complaints': 'number_of_open_complaints',
        'Policy Type': 'policy_type',
        'Vehicle Class': 'vehicle_class',
        'Total Claim Amount': 'total_claim_amount'
    })
    
    # Convert customer lifetime value to numeric
    df['customer_lifetime_value'] = (
        df['customer_lifetime_value']
        .astype(str)
        .str.replace('%', '', regex=False)
    )
    df['customer_lifetime_value'] = pd.to_numeric(
        df['customer_lifetime_value'], 
        errors='coerce'
    )
    
    # Convert open complaints
    df['number_of_open_complaints'] = (
        df['number_of_open_complaints']
        .astype(str)
        .str.split('/')
        .str[0]
    )
    df['number_of_open_complaints'] = pd.to_numeric(
        df['number_of_open_complaints'],
        errors='coerce'
    )
    
    # Remove duplicates
    df = df.drop_duplicates()
    
    # Reset index
    df = df.reset_index(drop=True)
    
    return df

In [11]:
df = clean_data(df)

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [13]:
filtered_customers = df[
    (df['total_claim_amount'] < 1000) &
    (df['Response'] == 'Yes')
]

filtered_customers

,Unnamed: 0,Customer,State,customer_lifetime_value,Response,Coverage,Education,Effective To Date,EmploymentStatus,Gender,...,number_of_open_complaints,Number of Policies,policy_type,Policy,Renew Offer Type,Sales Channel,total_claim_amount,vehicle_class,Vehicle Size,Vehicle Type
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
8,8,FM55990,California,5989.773931,Yes,Premium,College,1/19/11,Employed,M,...,0.0,1,Personal Auto,Personal L1,Offer2,Branch,739.200000,Sports Car,Medsize,NaN
15,15,CW49887,California,4626.801093,Yes,Basic,Master,1/16/11,Employed,F,...,0.0,1,Special Auto,Special L1,Offer2,Branch,547.200000,SUV,Medsize,NaN
19,19,NJ54277,California,3746.751625,Yes,Extended,College,2/26/11,Employed,F,...,1.0,1,Personal Auto,Personal L2,Offer2,Call Center,19.575683,Two-Door Car,Large,A
27,27,MQ68407,Oregon,4376.363592,Yes,Premium,Bachelor,2/28/11,Employed,F,...,0.0,1,Personal Auto,Personal L3,Offer2,Agent,60.036683,Four-Door Car,Medsize,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10844,10844,FM31768,Arizona,5979.724161,Yes,Extended,High School or Below,2/7/11,Employed,F,...,0.0,3,Personal Auto,Personal L1,Offer2,Agent,547.200000,Four-Door Car,Medsize,NaN
10852,10852,KZ80424,Washington,8382.478392,Yes,Basic,Bachelor,1/27/11,Employed,M,...,0.0,2,Personal Auto,Personal L2,Offer2,Call Center,791.878042,NaN,NaN,A
10872,10872,XT67997,California,5979.724161,Yes,Extended,High School or Below,2/7/11,Employed,F,...,0.0,3,Personal Auto,Personal L3,Offer2,Agent,547.200000,Four-Door Car,Medsize,NaN
10887,10887,BY78730,Oregon,8879.790017,Yes,Basic,High School or Below,2/3/11,Employed,F,...,0.0,7,Special Auto,Special L2,Offer1,Agent,528.200860,SUV,Small,A


2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [15]:
yes_customers = df[df['Response'] == 'Yes']

analysis = yes_customers.groupby(['policy_type', 'Gender'])[
    ['monthly_premium_auto', 'customer_lifetime_value', 'total_claim_amount']
].mean()

analysis

monthly_premium_auto  customer_lifetime_value  \
policy_type    Gender                                                  
Corporate Auto F                  94.301775              7712.628736   
               M                  92.188312              7944.465414   
Personal Auto  F                  98.998148              8339.791842   
               M                  91.085821              7448.383281   
Special Auto   F                  92.314286              7691.584111   
               M                  86.343750              8247.088702   

                       total_claim_amount  
policy_type    Gender                      
Corporate Auto F               433.738499  
               M               408.582459  
Personal Auto  F               452.965929  
               M               457.010178  
Special Auto   F               453.280164  
               M               429.527942

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [17]:
customers_by_state = df['State'].value_counts()

customers_by_state

State
California    3552
Oregon        2909
Arizona       1937
Nevada         993
Washington     888
Name: count, dtype: int64

In [18]:
customers_by_state[customers_by_state > 500]

State
California    3552
Oregon        2909
Arizona       1937
Nevada         993
Washington     888
Name: count, dtype: int64

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [20]:
clv_by_education_gender = df.groupby(['Education', 'Gender'])['customer_lifetime_value'].agg(
    ['max', 'min', 'median']
)

clv_by_education_gender

max          min       median
Education            Gender                                       
Bachelor             F       73225.95652  1904.000852  5640.505303
                     M       67907.27050  1898.007675  5548.031892
College              F       61850.18803  1898.683686  5623.611187
                     M       61134.68307  1918.119700  6005.847375
Doctor               F       44856.11397  2395.570000  5332.462694
                     M       32677.34284  2267.604038  5577.669457
High School or Below F       55277.44589  2144.921535  6039.553187
                     M       83325.38119  1940.981221  6286.731006
Master               F       51016.06704  2417.777032  5729.855012
                     M       50568.25912  2272.307310  5579.099207

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

In [27]:
df['month'] = pd.to_datetime(df['Effective To Date']).dt.month

policies_by_state_month = pd.pivot_table(
    df,
    index='State',
    columns='month',
    values='Number of Policies',
    aggfunc='count'
)

policies_by_state_month

C:\Users\Raquel Marques\AppData\Local\Temp\ipykernel_1220\15990863.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['month'] = pd.to_datetime(df['Effective To Date']).dt.month


month,1,2
State,,
Arizona,1008,929
California,1918,1634
Nevada,551,442
Oregon,1565,1344
Washington,463,425


6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

In [29]:
policies_by_state_month = df.groupby(['State', 'month'])['Number of Policies'].count().reset_index(name='policy_count')

In [31]:
top_3_states = (
    policies_by_state_month.groupby('State')['policy_count']
    .sum()
    .sort_values(ascending=False)
    .head(3)
)

In [33]:
top_3_by_month = policies_by_state_month[
    policies_by_state_month['State'].isin(top_3_states.index)
]

In [34]:
top_3_by_month

,State,month,policy_count
0,Arizona,1,1008
1,Arizona,2,929
2,California,1,1918
3,California,2,1634
6,Oregon,1,1565
7,Oregon,2,1344


7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [37]:
response_by_channel = df['Sales Channel'].value_counts()
response_by_channel

Sales Channel
Agent          4121
Branch         3022
Call Center    2141
Web            1626
Name: count, dtype: int64

In [38]:
response_rate = (
    df.groupby('Sales Channel')['Response']
    .apply(lambda x: (x == 'Yes').mean() * 100)
    .reset_index(name='response_rate')
)

response_rate

,Sales Channel,response_rate
0,Agent,18.005339
1,Branch,10.787558
2,Call Center,10.322279
3,Web,10.885609
